In [ ]:
import torch

SEED = 1234
torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True

# 0. Choosing "teacher" model

In [ ]:
checkpoint = "bert-base-uncased"
task_name  = "qqp"

# 1. Loading our mrpc part of the GLUE dataset

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer, DataCollatorWithPadding

raw_datasets = load_dataset("glue", task_name)
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

In [ ]:
raw_datasets

In [ ]:
raw_datasets['train']

In [ ]:
from datasets import DatasetDict

In [ ]:
raw_datasets = DatasetDict({
    "train": raw_datasets['train'],
    "validation": raw_datasets['validation'],
    "test": raw_datasets['test']
})

In [ ]:
raw_datasets

In [ ]:
raw_train_dataset = raw_datasets["train"]
raw_train_dataset[0]

In [ ]:
raw_train_dataset[5]['question1'], raw_train_dataset[5]['question2']

In [ ]:
raw_train_dataset[5]['label']

In [ ]:
raw_train_dataset[5]['idx']

In [ ]:
raw_train_dataset.features

# 2. Preprocess

In [ ]:
def tokenize_function(example):
    return tokenizer(example["question1"], example["question2"], truncation=True)

tokenized_datasets = raw_datasets.map(tokenize_function, batched=True)
tokenized_datasets

# 3. Preparing for Training

In [ ]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [ ]:
tokenized_datasets = tokenized_datasets.remove_columns(["question1", "question2", "idx"])
tokenized_datasets = tokenized_datasets.rename_column("label", "labels")
tokenized_datasets.set_format("torch")
tokenized_datasets["train"].column_names

In [ ]:
from torch.utils.data import DataLoader

train_dataloader = DataLoader(
    tokenized_datasets["train"], shuffle=True, batch_size=32, collate_fn=data_collator
)

val_dataloader = DataLoader(
    tokenized_datasets["validation"], batch_size=32, collate_fn=data_collator
)

eval_dataloader = DataLoader(
    tokenized_datasets["test"], batch_size=32, collate_fn=data_collator
)

In [ ]:
for batch in train_dataloader:
    break
{k: v.shape for k, v in batch.items()}

# 4. Loading model

In [ ]:
import sys
sys.path.insert(0, '../')

In [ ]:
from Bert_model.modeling_bert import BertForSequenceClassification

In [ ]:
# id2label, label2id dicts for the outputs for the model
labels = tokenized_datasets["train"].features["labels"].names
num_labels = len(labels)
label2id, id2label = dict(), dict()
for i, label in enumerate(labels):
    label2id[label] = str(i)
    id2label[str(i)] = label

In [ ]:
model = BertForSequenceClassification.from_pretrained(
    checkpoint,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
    output_hidden_states=True,
    output_attentions=True
)

In [ ]:
model.set_use_module_grafting(False)
model.set_use_scc_status(False)

In [ ]:
outputs = model(**batch)
print(outputs.loss, outputs.logits.shape)

In [ ]:
device = torch.device("cuda:1") if torch.cuda.is_available() else torch.device("cpu")
model.to(device)

device

### Load Trained Weights

In [ ]:
load_path = '../glue_fine_tune/weights/'
best_weight = torch.load(load_path + f'bert-{task_name}.pt', map_location=device)
model.load_state_dict(best_weight['model_state_dict'])

In [ ]:
from train_eval_func import eval_loop

In [ ]:
eval_loop(model, val_dataloader, task_name, device)[0]

### Load Trained Weights for Model with 8 Layers

In [ ]:
import os
from merge_helper import merge_bert_layers, LayerMergeTracker, apply_layer_tracking_to_model
from train_eval_func import train, get_primary_metric, set_lr_scheduler, get_recovery_epoch
from torch.optim import AdamW

In [ ]:
model_8 = BertForSequenceClassification.from_pretrained(
    checkpoint,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
    output_hidden_states=True,
    output_attentions=True
)

In [ ]:
load_graft_path = '../similar_layer_merge/weights/'
graft_state = torch.load(load_graft_path + f'layer-8-{task_name}.pt', map_location=device)

In [ ]:
model_8 = apply_layer_tracking_to_model(model_8, graft_state['layer_track'])

In [ ]:
len(model_8.bert.encoder.layer)

In [ ]:
model_8.to(device)
device

In [ ]:
model_8.load_state_dict(graft_state['model_state_dict'])

In [ ]:
eval_loop(model_8, val_dataloader, task_name, device)[0]

### Load Trained Weights for Model with 7 Layers

In [ ]:
model_7 = BertForSequenceClassification.from_pretrained(
    checkpoint,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
    output_hidden_states=True,
    output_attentions=True
)

In [ ]:
load_graft_path = '../similar_layer_merge/weights/'
graft_state = torch.load(load_graft_path + f'layer-7-{task_name}.pt', map_location=device)

In [ ]:
model_7 = apply_layer_tracking_to_model(model_7, graft_state['layer_track'])

In [ ]:
len(model_7.bert.encoder.layer)

In [ ]:
model_7.to(device)
device

In [ ]:
model_7.load_state_dict(graft_state['model_state_dict'])

In [ ]:
eval_loop(model_7, val_dataloader, task_name, device)[0]

### Load Trained Weights for Model with 6 Layers

In [ ]:
model_6 = BertForSequenceClassification.from_pretrained(
    checkpoint,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
    output_hidden_states=True,
    output_attentions=True
)

In [ ]:
load_graft_path = '../similar_layer_merge/weights/'
graft_state = torch.load(load_graft_path + f'layer-6-{task_name}.pt', map_location=device)

In [ ]:
model_6 = apply_layer_tracking_to_model(model_6, graft_state['layer_track'])

In [ ]:
len(model_6.bert.encoder.layer)

In [ ]:
model_6.to(device)
device

In [ ]:
model_6.load_state_dict(graft_state['model_state_dict'])

In [ ]:
eval_loop(model_6, val_dataloader, task_name, device)[0]

In [ ]:
print(model.bert.encoder.layer[0].state_dict().keys())

# 5. Pruning Sweep — Depth Model vs Width Model

We compare how **two structurally different fine-tuned BERT variants** respond to
unstructured L1 weight pruning at five sparsity levels: **10%, 20%, 30%, 40%, 50%**.

| Model | Structure |
|---|---|
| `model` | Standard BERT-base (12 layers, depth) |
| `graft_model` | Width-merged BERT (6 layers, 2× internal width) |

### GPU-efficient strategy
- Both models stay on GPU throughout — no `.to(device)` inside the loop
- Original dense weights are cached **once** as CPU tensors
- Between pruning levels, weights are restored via a CPU→GPU `load_state_dict` copy
- Stale masks are stripped with `remove_all_masks()` before each new level


In [ ]:
from prune_helper import remove_all_masks, apply_pruning, global_sparsity, make_permanent, compare_pruning_robustness, print_pruning_results, plot_pruning_comparison

In [ ]:
# Cache both models' dense weights on CPU — cheap to store, cheap to restore
depth_state_cpu = {k: v.cpu().clone() for k, v in model.state_dict().items()}
width_state_8_cpu = {k: v.cpu().clone() for k, v in model_8.state_dict().items()}
width_state_7_cpu = {k: v.cpu().clone() for k, v in model_7.state_dict().items()}
width_state_6_cpu = {k: v.cpu().clone() for k, v in model_6.state_dict().items()}

print(f'Depth model: {len(depth_state_cpu)} tensors cached on CPU')
print(f'Width model (8 layers): {len(width_state_8_cpu)} tensors cached on CPU')
print(f'Width model (7 layers): {len(width_state_7_cpu)} tensors cached on CPU')
print(f'Width model (6 layers): {len(width_state_6_cpu)} tensors cached on CPU')
print(f'GPU device : {device}')


In [ ]:
model.eval()
model_8.eval()
model_7.eval()
model_6.eval()

baseline_depth, _ = eval_loop(model, val_dataloader, task_name, device)
baseline_width_8, _ = eval_loop(model_8, val_dataloader, task_name, device)
baseline_width_7, _ = eval_loop(model_7, val_dataloader, task_name, device)
baseline_width_6, _ = eval_loop(model_6, val_dataloader, task_name, device)

print(f'Baseline — Depth model (12-layer)')
for metric_name, value in baseline_depth.items():
    print(f"- {metric_name}: {value:.4f}")

print(f'Baseline — Width model (8-layer)')
for metric_name, value in baseline_width_8.items():
    print(f"- {metric_name}: {value:.4f}")

print(f'Baseline — Width model (7-layer)')
for metric_name, value in baseline_width_7.items():
    print(f"- {metric_name}: {value:.4f}")

print(f'Baseline — Width model (6-layer)')
for metric_name, value in baseline_width_6.items():
    print(f"- {metric_name}: {value:.4f}")


In [ ]:
# Pass models with baselines
models = {
    '12-layer': (model, baseline_depth),
    '8-layer': (model_8, baseline_width_8),
    '7-layer': (model_7, baseline_width_7),
    '6-layer': (model_6, baseline_width_6),
}

In [ ]:
results = compare_pruning_robustness(
    models_dict=models,
    eval_dataloader=val_dataloader,  # or eval_dataloader
    task_name=task_name,
    device=device,
    pruning_levels=[0.10, 0.20, 0.30, 0.40, 0.50],
    save_checkpoints=False
)

In [ ]:
# Print results
print_pruning_results(models, results)

In [ ]:
plot_pruning_comparison(results_dict=results)

# 6. Global Pruning — Attention vs FFN Separately

**Local pruning** (Section 5) removes the lowest X% of weights *within each layer*,
treating every layer equally regardless of how important its weights are.

**Global pruning** ranks weights *across all layers of the same type* and removes
the globally weakest ones — so some layers get pruned more than others based on
their actual weight magnitudes. This is generally smarter.

We split the model into **two global pools**:

| Pool | Layers included |
|---|---|
| **Attention** | `query`, `key`, `value`, `attention.output.dense` across all encoder layers |
| **FFN** | `intermediate.dense`, `output.dense` across all encoder layers |

This lets you control attention sparsity and FFN sparsity independently —
e.g. prune FFN aggressively (it's 2/3 of BERT's parameters) while being
gentler on attention.


In [ ]:
from prune_helper import get_global_pruning_params, apply_global_pruning, pool_sparsity, remove_global_masks, remove_all_global_masks, describe_pools, print_per_layer_sparsity, compare_global_pruning_robustness

## Inspect Pool Sizes

In [ ]:
describe_pools(model,  'Depth model (12-layer standard BERT)')
describe_pools(model_8, 'Width model (8-layer merged BERT)')
describe_pools(model_7, 'Width model (7-layer merged BERT)')
describe_pools(model_6, 'Width model (6-layer merged BERT)')

## Global Pruning Sweep¶

In [ ]:
results = compare_global_pruning_robustness(
    models_dict=models,
    eval_dataloader=val_dataloader,
    task_name=task_name,
    device=device,
    save_checkpoints=False
)

In [ ]:
results = compare_global_pruning_robustness(
    models_dict=models,
    eval_dataloader=val_dataloader,
    task_name=task_name,
    device=device,
    save_checkpoints=False,
    pruning_configs=[
        (0, 0.2),  # FFN-only 20%
        (0, 0.3),  # FFN-only 30%
        (0, 0.4),  # FFN-only 40%
        (0, 0.5),  # FFN-only 50%
    ]
)

In [ ]:
results = compare_global_pruning_robustness(
    models_dict=models,
    eval_dataloader=val_dataloader,
    task_name=task_name,
    device=device,
    save_checkpoints=False,
    pruning_configs=[
        (0.2, 0),  # Attn-only 20%
        (0.3, 0),  # Attn-only 30%
        (0.4, 0),  # Attn-only 40%
        (0.5, 0),  # Attn-only 50%
    ]
)